# Notebook 05 — Matrix Factorization: SVD & ALS

**Purpose:** Implement and evaluate two complementary matrix factorization models:
- **FunkSVD** — biased MF via SGD, optimised for RMSE / MAE
- **ALS** — implicit confidence-weighted, optimised for Precision@K / NDCG@K

**Inputs:**
- `data/splits/train.csv`
- `data/splits/test.csv`
- `data/processed/index_maps.pkl`
- `results/baseline_model.pkl`

**Outputs:**
- `results/svd_model.pkl`
- `results/als_model.pkl`
- Rating metrics: RMSE, MAE
- Ranking metrics: Precision@10, NDCG@10

## 0. Imports & Configuration

In [4]:
import sys, os, time
import numpy as np
import pandas as pd
import joblib

sys.path.insert(0, os.path.abspath("../src"))
from matix_factorization import SVDModel, ALSModel, svd_grid_search
from evaluation import evaluate_rating_predictions, run_sanity_checks

# ── Paths ────────────────────────────────────────────────
TRAIN_PATH    = "../data/splits/train.csv"
TEST_PATH     = "../data/splits/test.csv"
MAPS_PATH     = "../data/processed/index_maps.pkl"
BASELINE_PATH = "../results/baseline_model.pkl"
RESULTS_DIR   = "../results/"

# ── Hyperparameter search space ──────────────────────────
# Search grid evaluated on a 10% temporal hold-out within train
SVD_GRID = {
    "n_factors": [50, 100],
    "n_epochs":  [20, 30],
    "lr":        [0.005],
    "reg":       [0.02, 0.05],
}

# ALS hyperparameters (pre-selected for dataset characteristics)
ALS_PARAMS = {"n_factors": 100, "reg": 0.1, "alpha": 15, "n_iters": 15}

# ── Evaluation settings ──────────────────────────────────
TOP_K         = 10
RELEVANCE_THR = 3.5    # Min rating to count a test item as relevant
RANK_SAMPLES  = 500    # Users sampled for ranking metrics
RMSE_RANGE    = (0.8, 1.5)
MAE_RANGE     = (0.6, 1.2)

# Prior model results
BASELINE_RMSE = 0.9047
CF_RMSE       = 0.9184
BASELINE_P10  = 0.0366
CF_P10        = 0.0838
CF_NDCG       = 0.100771

os.makedirs(RESULTS_DIR, exist_ok=True)
print("Configuration loaded.")

Configuration loaded.


## 1. Load Data

Load train, test, and index maps from disk. The matrix is not reloaded — we train SVD/ALS directly from the train DataFrame to preserve maximum flexibility during hyperparameter tuning.

In [5]:
train    = pd.read_csv(TRAIN_PATH)
test     = pd.read_csv(TEST_PATH)
maps     = joblib.load(MAPS_PATH)
u2i      = maps["user_to_idx"]
m2i      = maps["movie_to_idx"]
baseline = joblib.load(BASELINE_PATH)

print(f"Train : {len(train):,} rows | {train['userId'].nunique():,} users")
print(f"Test  : {len(test):,} rows")
print(f"Users : {len(u2i):,} | Items: {len(m2i):,}")

Train : 1,565,182 rows | 12,773 users
Test  : 397,622 rows
Users : 12,773 | Items: 14,246


## 2. SVD — Hyperparameter Grid Search

Grid search over `n_factors`, `n_epochs`, `lr`, and `reg` using the last 10% of each user's train history as a validation set. This respects the temporal split — no test data is ever touched.

In [6]:
print("SVD Grid Search (10% temporal hold-out within train)...")
t0  = time.time()
gs  = svd_grid_search(train, u2i, m2i, baseline, SVD_GRID,
                       n_val_ratio=0.1, verbose=True)
bp  = gs["best_params"]
print(f"\nBest params    : {bp}")
print(f"Best val RMSE  : {gs['best_val_rmse']:.4f}")
print(f"Grid search time: {time.time()-t0:.1f}s")

print("\nAll results:")
for r in sorted(gs["results"], key=lambda x: x["val_rmse"]):
    print(f"  nf={r['params']['n_factors']:>3}  ep={r['params']['n_epochs']:>2}  "
          f"reg={r['params']['reg']}  val_rmse={r['val_rmse']:.4f}")

SVD Grid Search (10% temporal hold-out within train)...
[svd_grid_search] tr=1,403,203  val=161,979
  Trying {'n_factors': 50, 'n_epochs': 20, 'lr': 0.005, 'reg': 0.02} ... [SVDModel.fit] Done. Final train-RMSE=0.7885
val_rmse=0.8295
  Trying {'n_factors': 50, 'n_epochs': 20, 'lr': 0.005, 'reg': 0.05} ... [SVDModel.fit] Done. Final train-RMSE=0.8275
val_rmse=0.8488
  Trying {'n_factors': 50, 'n_epochs': 30, 'lr': 0.005, 'reg': 0.02} ... [SVDModel.fit] Done. Final train-RMSE=0.7185
val_rmse=0.8074
  Trying {'n_factors': 50, 'n_epochs': 30, 'lr': 0.005, 'reg': 0.05} ... [SVDModel.fit] Done. Final train-RMSE=0.7957
val_rmse=0.8307
  Trying {'n_factors': 100, 'n_epochs': 20, 'lr': 0.005, 'reg': 0.02} ... [SVDModel.fit] Done. Final train-RMSE=0.7786
val_rmse=0.8259
  Trying {'n_factors': 100, 'n_epochs': 20, 'lr': 0.005, 'reg': 0.05} ... [SVDModel.fit] Done. Final train-RMSE=0.8244
val_rmse=0.8471
  Trying {'n_factors': 100, 'n_epochs': 30, 'lr': 0.005, 'reg': 0.02} ... [SVDModel.fit] Done.

## 3. SVD — Retrain on Full Train Set

Using the best hyperparameters, retrain FunkSVD on the complete training data. More data → better generalization. Convergence is monitored via train-RMSE.

In [7]:
svd = SVDModel(
    baseline=baseline,
    n_factors=bp["n_factors"],
    n_epochs=bp["n_epochs"],
    lr=bp["lr"],
    reg=bp["reg"],
)
t1 = time.time()
svd.fit(train, u2i, m2i, verbose=True)
print(f"Full-train fit time: {time.time()-t1:.1f}s")

[SVDModel.fit] 12,773u x 14,246i factors=100 epochs=30 lr=0.005 reg=0.02
  Epoch   1/30  train-RMSE=0.9306
  Epoch   2/30  train-RMSE=0.8813
  Epoch   3/30  train-RMSE=0.8692
  Epoch   4/30  train-RMSE=0.8630
  Epoch   5/30  train-RMSE=0.8591
  Epoch   6/30  train-RMSE=0.8564
  Epoch   7/30  train-RMSE=0.8542
  Epoch   8/30  train-RMSE=0.8522
  Epoch   9/30  train-RMSE=0.8497
  Epoch  10/30  train-RMSE=0.8459
  Epoch  11/30  train-RMSE=0.8403
  Epoch  12/30  train-RMSE=0.8332
  Epoch  13/30  train-RMSE=0.8258
  Epoch  14/30  train-RMSE=0.8184
  Epoch  15/30  train-RMSE=0.8108
  Epoch  16/30  train-RMSE=0.8031
  Epoch  17/30  train-RMSE=0.7953
  Epoch  18/30  train-RMSE=0.7873
  Epoch  19/30  train-RMSE=0.7795
  Epoch  20/30  train-RMSE=0.7716
  Epoch  21/30  train-RMSE=0.7637
  Epoch  22/30  train-RMSE=0.7557
  Epoch  23/30  train-RMSE=0.7478
  Epoch  24/30  train-RMSE=0.7399
  Epoch  25/30  train-RMSE=0.7319
  Epoch  26/30  train-RMSE=0.7238
  Epoch  27/30  train-RMSE=0.7157
  Epoch  

## 4. SVD — Evaluation

Vectorised batch prediction on all test rows, then compute RMSE and MAE. Unknown (userId, movieId) pairs fall back to the biased baseline automatically.

In [8]:
y_pred_svd, svd_src = svd.predict_batch(test)
rm_svd = evaluate_rating_predictions(test, y_pred_svd)

print("=" * 50)
print("  SVD RATING METRICS")
print("=" * 50)
print(f"  RMSE  : {rm_svd['rmse']:.4f}  (baseline {BASELINE_RMSE:.4f}  delta {rm_svd['rmse']-BASELINE_RMSE:+.4f})")
print(f"  MAE   : {rm_svd['mae']:.4f}")
print(f"  NaN   : {rm_svd['n_nan_preds']}")
print(f"  SVD coverage   : {100*svd_src['svd']/len(test):.1f}%")
print(f"  Baseline used  : {svd_src['baseline']:,} rows")

# Sanity checks
checks = run_sanity_checks(y_pred_svd, rm_svd["rmse"], rm_svd["mae"], RMSE_RANGE, MAE_RANGE)
print("\nSanity Checks:")
for c in checks:
    print(f"  [{'PASS' if c['passed'] else 'FAIL'}] {c['check']} -- {c['detail']}")
assert all(c["passed"] for c in checks), "SVD sanity check failed"
print("\nAll SVD sanity checks passed.")

joblib.dump(svd, RESULTS_DIR + "svd_model.pkl")
print("SVD model saved.")

  SVD RATING METRICS
  RMSE  : 0.8193  (baseline 0.9047  delta -0.0854)
  MAE   : 0.6146
  NaN   : 0
  SVD coverage   : 99.9%
  Baseline used  : 321 rows

Sanity Checks:
  [PASS] No NaN predictions -- 0 NaN values found
  [PASS] All predictions in [0.5, 5.0] -- 0 out-of-range values
  [PASS] RMSE in (0.8, 1.5) -- RMSE = 0.8193
  [PASS] MAE in (0.6, 1.2) -- MAE = 0.6146

All SVD sanity checks passed.
SVD model saved.


## 5. ALS — Train on Implicit Feedback

Convert explicit ratings to binary preferences and confidence weights, then train ALS. ALS is optimised for ranking rather than point-wise rating accuracy.

In [9]:
als = ALSModel(
    baseline=baseline,
    n_factors=ALS_PARAMS["n_factors"],
    reg=ALS_PARAMS["reg"],
    alpha=ALS_PARAMS["alpha"],
    n_iters=ALS_PARAMS["n_iters"],
    preference_threshold=RELEVANCE_THR,
)
t2 = time.time()
als.fit(train, u2i, m2i, verbose=True)
print(f"\nALS fit time: {time.time()-t2:.1f}s")

joblib.dump(als, RESULTS_DIR + "als_model.pkl")
print("ALS model saved.")

[ALSModel.fit] 12,773u x 14,246i factors=100 iters=15 alpha=15 reg=0.1
  Iter   1/15  loss=12351258.01
  Iter   2/15  loss=6306125.13
  Iter   3/15  loss=5514640.82
  Iter   4/15  loss=5152877.88
  Iter   5/15  loss=4940120.69
  Iter   6/15  loss=4796998.88
  Iter   7/15  loss=4692202.04
  Iter   8/15  loss=4610919.75
  Iter   9/15  loss=4545242.65
  Iter  10/15  loss=4490611.67
  Iter  11/15  loss=4444226.28
  Iter  12/15  loss=4404243.00
  Iter  13/15  loss=4369363.85
  Iter  14/15  loss=4338625.81
  Iter  15/15  loss=4311290.82
[ALSModel.fit] Done.

ALS fit time: 1097.7s
ALS model saved.


## 6. ALS — Ranking Evaluation

Sample 500 users with at least one relevant test item. For each user, generate Top-10 recommendations (seen items masked), then compute Precision@10 and NDCG@10.

In [10]:
train_seen = train.groupby("userId")["movieId"].apply(set).to_dict()
rel_users  = test[test["rating"] >= RELEVANCE_THR]["userId"].unique()
rng        = np.random.default_rng(42)
sampled    = rng.choice(rel_users, size=min(RANK_SAMPLES, len(rel_users)), replace=False)

precs, ndcgs = [], []
for uid in sampled:
    ut  = test[test["userId"] == uid]
    rel = set(ut.loc[ut["rating"] >= RELEVANCE_THR, "movieId"])
    if not rel:
        continue
    seen = train_seen.get(uid, set())
    recs = als.recommend_top_k(uid, k=TOP_K, seen_items=seen)

    # Verify masking
    assert not (set(recs) & seen), f"Seen item leaked for user {uid}"

    hits = sum(1 for m in recs if m in rel)
    precs.append(hits / TOP_K)
    dcg = sum(1.0/np.log2(i+2) for i,m in enumerate(recs) if m in rel)
    idl = sum(1.0/np.log2(i+2) for i in range(min(len(rel), TOP_K)))
    ndcgs.append(dcg/idl if idl else 0.0)

p10 = float(np.mean(precs))
n10 = float(np.mean(ndcgs))

print("=" * 50)
print("  ALS RANKING METRICS")
print("=" * 50)
print(f"  Precision@{TOP_K} : {p10:.4f}  (CF: {CF_P10:.4f}  delta: {p10-CF_P10:+.4f})")
print(f"  NDCG@{TOP_K}      : {n10:.4f}  (CF: {CF_NDCG:.4f}  delta: {n10-CF_NDCG:+.4f})")
print(f"  Users evaluated  : {len(precs):,}")
print(f"  Masking verified : OK (assert passed)")

  ALS RANKING METRICS
  Precision@10 : 0.0788  (CF: 0.0838  delta: -0.0050)
  NDCG@10      : 0.1037  (CF: 0.1008  delta: +0.0030)
  Users evaluated  : 500
  Masking verified : OK (assert passed)


## 7. Final Comparison Table

Summarise all four models across both evaluation objectives.

In [11]:
print("=" * 65)
print(f"  {'Model':<12} {'RMSE':>8} {'MAE':>8} {'Prec@10':>10} {'NDCG@10':>10}")
print("-" * 65)
print(f"  {'Baseline':<12} {BASELINE_RMSE:>8.4f} {'0.6826':>8} {BASELINE_P10:>10.4f} {'N/A':>10}")
print(f"  {'CF':<12} {CF_RMSE:>8.4f} {'0.6753':>8} {CF_P10:>10.4f} {CF_NDCG:>10.4f}")
print(f"  {'SVD ★':>12} {rm_svd['rmse']:>8.4f} {rm_svd['mae']:>8.4f} {'N/A':>10} {'N/A':>10}")
print(f"  {'ALS':>12} {'N/A':>8} {'N/A':>8} {p10:>10.4f} {n10:>10.4f}")
print("=" * 65)
print(f"\n  ★ Best RMSE: SVD ({rm_svd['rmse']:.4f}) — beats baseline by {BASELINE_RMSE - rm_svd['rmse']:+.4f}")
print(f"  ★ Best NDCG@10: ALS ({n10:.4f}) — beats CF by {n10 - CF_NDCG:+.4f}")

  Model            RMSE      MAE    Prec@10    NDCG@10
-----------------------------------------------------------------
  Baseline       0.9047   0.6826     0.0366        N/A
  CF             0.9184   0.6753     0.0838     0.1008
         SVD ★   0.8193   0.6146        N/A        N/A
           ALS      N/A      N/A     0.0788     0.1037

  ★ Best RMSE: SVD (0.8193) — beats baseline by +0.0854
  ★ Best NDCG@10: ALS (0.1037) — beats CF by +0.0030
